# 04 - Chunking

RAG stores pieces of documents, called chunks, and retrieves the best few for
each question. This notebook explains why chunking is needed and builds a
chunking function by hand. That same function feeds our vector database in
notebook 05.

**What you will learn**

- Why documents must be split into chunks
- What tokens are and how to count them with tiktoken
- How to write a splitter with overlap
- The trade-off between small and large chunks

## Why chunk at all?

Two reasons:

1. **Precision.** An embedding squeezes a whole text into one vector. Embed a
   30-page document and the vector becomes a blurry average of every topic in
   it. Embed one focused paragraph and the vector is sharp. Sharp vectors
   make search accurate.
2. **Prompt budget.** Retrieved chunks get pasted into the prompt. Prompts
   have a size limit, and you pay per token. Small chunks let you include
   several relevant pieces instead of one giant document.

## Tokens: how LLMs measure text

Models do not read characters or words. They read tokens: common fragments of
text. As a rule of thumb, one token is about 4 characters or three quarters
of a word in English.

Tokens matter to you because context limits and pricing are both counted in
tokens. The tiktoken library shows exactly how text becomes tokens:

In [1]:
import tiktoken

encoding = tiktoken.encoding_for_model("gpt-4o-mini")

text = "The Carrier X2 delivers medicine autonomously."
token_ids = encoding.encode(text)

print("Token count:", len(token_ids))
print("The pieces: ", [encoding.decode([t]) for t in token_ids])

Token count: 9
The pieces:  ['The', ' Carrier', ' X', '2', ' delivers', ' medicine', ' autonom', 'ously', '.']


Common words are one token, rarer words split into fragments. Let us
count tokens in our sample documents:

In [2]:
from pathlib import Path

def count_tokens(text):
    return len(encoding.encode(text))

for path in sorted(Path("../data").glob("*.txt")):
    text = path.read_text()
    print(f"{count_tokens(text):>5} tokens  {path.name}")

  194 tokens  01-company-overview.txt
  254 tokens  02-product-guide.txt
  216 tokens  03-leave-policy.txt
  213 tokens  04-remote-work-policy.txt
  317 tokens  05-support-runbook.txt
  268 tokens  06-q1-2025-update.txt


Our documents are tiny (a real knowledge base would have documents of
thousands of tokens each), but they are big enough to practice on.

## A chunking function

The simplest solid strategy is **fixed size with overlap**:

- cut the text every N words
- let each chunk share its last few words with the next chunk

The overlap matters. A sentence that would be cut in half at a boundary
appears intact in one of the two neighboring chunks, so its meaning is never
lost to search.

In [3]:
def chunk_text(text, chunk_size=150, overlap=30):
    """Split text into chunks of about chunk_size words.
    Consecutive chunks share overlap words."""
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunks.append(" ".join(words[start:end]))
        if end >= len(words):
            break
        start = end - overlap
    return chunks

In [4]:
text = Path("../data/03-leave-policy.txt").read_text()
chunks = chunk_text(text, chunk_size=80, overlap=20)

print(f"The document became {len(chunks)} chunks.\n")
for i, chunk in enumerate(chunks):
    print(f"--- chunk {i} ({count_tokens(chunk)} tokens) ---")
    print(chunk[:200] + "...\n")

The document became 3 chunks.

--- chunk 0 (96 tokens) ---
Aurora Dynamics - Employee Handbook: Leave Policy This policy applies to all full time employees of Aurora Dynamics. Annual leave: Every employee receives 24 days of paid annual leave per calendar yea...

--- chunk 1 (98 tokens) ---
Employees receive 12 days of paid sick leave per year. A doctor's note is required only for absences longer than 3 consecutive days. Parental leave: Primary caregivers receive 26 weeks of paid parenta...

--- chunk 2 (59 tokens) ---
public holiday calendar. In addition, the whole company closes for a shared winter break from December 25 to January 1. Requesting leave: All leave is requested through the internal tool called Compas...



Print two neighboring chunks in full and you will see the shared words
at the seam: the end of chunk 0 repeats as the start of chunk 1.

In [5]:
print("END OF CHUNK 0:\n...", chunks[0][-150:])
print()
print("START OF CHUNK 1:\n", chunks[1][:150], "...")

END OF CHUNK 0:
... 31 of the following year. Sick leave: Employees receive 12 days of paid sick leave per year. A doctor's note is required only for absences longer than

START OF CHUNK 1:
 Employees receive 12 days of paid sick leave per year. A doctor's note is required only for absences longer than 3 consecutive days. Parental leave: P ...


## Choosing a chunk size

There is no single right answer, only a trade-off:

| | Small chunks (50-150 words) | Large chunks (300-800 words) |
|---|---|---|
| Search precision | High, vectors are focused | Lower, vectors are blurry |
| Context for the LLM | May miss surrounding detail | Rich, keeps explanations whole |
| Prompt cost | Cheap | Expensive |

A common starting point is 100-300 words (roughly 150-400 tokens) with
10-20 percent overlap, then adjust based on the answers you get. Smarter
strategies exist (splitting on paragraphs, headings, or sentences), and they
are worth exploring later, but fixed size with overlap is the baseline that
every RAG system starts from.

## Chunking the whole corpus

Finally, chunk every document and keep track of where each chunk came from.
The source filename will become metadata in our vector database, which is
what lets the final system cite its sources.

In [6]:
all_chunks = []

for path in sorted(Path("../data").glob("*.txt")):
    for i, chunk in enumerate(chunk_text(path.read_text(), chunk_size=150, overlap=30)):
        all_chunks.append({"id": f"{path.stem}-{i}", "text": chunk, "source": path.name})

print(f"Corpus: {len(all_chunks)} chunks\n")
for c in all_chunks:
    print(f"  {c['id']:<28} {count_tokens(c['text']):>4} tokens")

Corpus: 11 chunks

  01-company-overview-0         176 tokens
  02-product-guide-0            194 tokens
  02-product-guide-1             85 tokens
  03-leave-policy-0             183 tokens
  03-leave-policy-1              59 tokens
  04-remote-work-policy-0       178 tokens
  04-remote-work-policy-1        62 tokens
  05-support-runbook-0          190 tokens
  05-support-runbook-1          155 tokens
  06-q1-2025-update-0           210 tokens
  06-q1-2025-update-1            94 tokens


## Exercise

1. Re-run the corpus cell with chunk_size=50. How many chunks do you get?
2. Try chunk_text with overlap=0 on the leave policy, and find a sentence
   that gets cut in half at a chunk boundary. That damaged sentence is
   exactly what overlap protects against.

In [7]:
# Try the exercise here
